# DSN Mart Sales Prediction

This Notebook is by Alegbeleye Opeyemi Micheal for the DSN Qualification Hackathon 2026 - ML Track.

**Library Requirements**
1. catboost
2. lightgbm
3. xgboost
4. ydata-profiling (only if you want the profile report)

**Contents**
1. Loading the data
2. Understanding the data
3. Testing my hypotheses
4. Encoding for models that need it
5. Data cleaning and feature engineering
6. CatBoost
7. LightGBM
8. Random Forest
9. XGBoost
10. Stacking
11. Summary of results

## 1. Loading the data

In [1]:
# Provided with both the train and test data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport

test_data = pd.read_csv('C:/Users/OPEYEMI/DSN-AI-BOOTCAMP-QUALIFICATION-HACKATHON/data/test.csv')
training_data = pd.read_csv('C:/Users/OPEYEMI/DSN-AI-BOOTCAMP-QUALIFICATION-HACKATHON/data/train.csv')

C:\Users\OPEYEMI\AppData\Local\Temp\ipykernel_44332\3485823968.py:6: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## 2. Understanding the data

**Quick structural check: column names, data types, a preview of rows, and summary statistics**

In [2]:
#to get a good view of the data

print(training_data.columns)
print(training_data.head())
print(training_data.info())
print(training_data.describe())

Index(['id', 'product_code', 'product_weight_kg', 'fat_content',
       'shelf_visibility', 'product_category', 'product_price', 'store_code',
       'store_age_years', 'store_size', 'store_location_tier', 'store_format',
       'total_sales'],
      dtype='object')
          id product_code  product_weight_kg fat_content  shelf_visibility  \
0  row_00000   PRD-PRFP9S             14.252     Low Fat            0.0271   
1  row_00001   PRD-PXXK71              7.698     Low Fat            0.0720   
2  row_00002   PRD-V5MOIJ             14.264     Regular            0.0421   
3  row_00003   PRD-UN5Z3J                NaN     Regular            0.0449   
4  row_00004   PRD-6RDQYB             10.338     Regular            0.0120   

     product_category  product_price store_code  store_age_years store_size  \
0        Frozen Foods          81.37  STORE-AGY               45      Large   
1  HEALTH AND HYGIENE          42.05  STORE-YLW               35      Small   
2              Canned      

To get better info on the dataset I use the "ydata_profiling" library. It is slow, so it is behind a switch. Set `run_profile = True` to see the report.

In [3]:
data_report = ProfileReport(training_data, minimal=True)
data_report.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 13/13 [00:00<00:00, 168.85it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
print(training_data.isna().sum())
print(test_data.isna().sum())

id                        0
product_code              0
product_weight_kg      1225
fat_content               0
shelf_visibility          0
product_category          0
product_price             0
store_code                0
store_age_years           0
store_size             1919
store_location_tier       0
store_format              0
total_sales               0
dtype: int64
id                       0
product_code             0
product_weight_kg      306
fat_content              0
shelf_visibility         0
product_category         0
product_price            0
store_code               0
store_age_years          0
store_size             491
store_location_tier      0
store_format             0
dtype: int64


Now I have a very good view of the data.

Categorical columns are
1. store_size
2. product_category
3. fat_content
4. store_code
5. store_location_tier
6. store_format

The columns **id, product_code, store_code, fat_content, shelf_visibility, product_category, product_price, store_age_years, total_sales, store_format and store_location_tier** have no missing values. The columns with missing values are **product_weight_kg and store_size**, and the test data has the same two. There are no duplicated ids.

## 3. Testing my hypotheses

In [5]:
#confirming suspicions based on personal inspection of the data..
#relationship between product code and product category

by_code = training_data.groupby('product_code')['product_category']
print('product codes with one category (raw labels)    :', round((by_code.nunique() == 1).mean(), 3))
print('product codes with one category (cleaned labels):',
      round((training_data['product_category'].str.lower().str.strip().groupby(training_data['product_code']).nunique() == 1).mean(), 3))
print('raw category labels:', training_data['product_category'].nunique())
print('cleaned category labels:', training_data['product_category'].str.lower().str.strip().nunique())
print('there are', training_data['product_code'].nunique(), 'unique product codes')
print('there are', training_data['store_code'].nunique(), 'unique store codes')

product codes with one category (raw labels)    : 0.105
product codes with one category (cleaned labels): 1.0
raw category labels: 48
cleaned category labels: 16
there are 1555 unique product codes
there are 10 unique store codes


Insights
1. A product code always belongs to one product category, but only once the letter casing is fixed. The 48 raw labels are really 16 categories ("Dairy", "DAIRY" and "dairy" are the same category). This has to be cleaned first.
2. There are only 10 unique store codes in the whole data.
3. With this I have a hypothesis that each product category will have a particular fat content.

In [6]:
display(training_data["total_sales"].describe())

plt.figure(figsize=(8, 5))
plt.hist(training_data["total_sales"], bins=40)
plt.title("Distribution of Total Sales")
plt.xlabel("total_sales")
plt.ylabel("Frequency")
plt.show()

count     6818.000000
mean      2174.756574
std       1697.894956
min         32.700000
25%        834.792500
50%       1790.890000
75%       3092.587500
max      12996.820000
Name: total_sales, dtype: float64

C:\Users\OPEYEMI\AppData\Local\Temp\ipykernel_44332\3962036914.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Total sales is right skewed. Most rows are below 3,000 but a few go as high as about 13,000.

In [7]:
#confirming my hypotheses
# firstly a mini data cleaning is done on only the product category column..
mini = training_data.copy()
mini['product_category'] = mini['product_category'].str.lower().str.strip()

pd.set_option('display.max_colwidth', None)
print(mini.groupby('fat_content')['product_category'].unique())

fat_content
Low Fat    [frozen foods, health and hygiene, soft drinks, canned, baking goods, household, snack foods, dairy, fruits and vegetables, meat, seafood, breakfast, breads, others, hard drinks, starchy foods]
Regular                                                        [canned, soft drinks, meat, snack foods, baking goods, dairy, fruits and vegetables, breakfast, frozen foods, breads, starchy foods, seafood]
Name: product_category, dtype: object


The hypothesis turned out to be false. The same product category can have different fat content.

**Another hypothesis**: "if the same product category has different fat content, how does it affect the product price?"

In [8]:
print(training_data.groupby('product_category')['fat_content'].value_counts().unstack())

fat_content            Low Fat  Regular
product_category                       
BAKING GOODS              67.0     70.0
BREADS                    29.0     16.0
BREAKFAST                  7.0     12.0
Baking Goods             126.0    128.0
Breads                    63.0     47.0
Breakfast                 14.0     27.0
CANNED                    63.0     54.0
Canned                   137.0    134.0
DAIRY                     97.0     46.0
Dairy                    168.0    101.0
FROZEN FOODS             100.0     72.0
FRUITS AND VEGETABLES    133.0    128.0
Frozen Foods             166.0    160.0
Fruits and Vegetables    245.0    235.0
HARD DRINKS               36.0      NaN
HEALTH AND HYGIENE        99.0      NaN
HOUSEHOLD                185.0      NaN
Hard Drinks               92.0      NaN
Health and Hygiene       197.0      NaN
Household                387.0      NaN
MEAT                      43.0     51.0
Meat                      65.0     92.0
OTHERS                    25.0      NaN


Categories representing non-food or ambiguous items: **household, health and hygiene, hard drinks, and others** show 100% Low Fat with zero Regular entries. This strongly suggests fat_content was defaulted for these rather than genuinely measured, since the real food categories all show a natural mix of both values.

In [9]:
print(training_data.groupby('product_category')['product_price'].agg(['mean', 'min', 'max']).round(1))
print(training_data.groupby('fat_content')['product_price'].mean().round(1))

                        mean   min    max
product_category                         
BAKING GOODS           124.8  36.1  264.2
BREADS                 125.9  31.6  263.0
BREAKFAST              138.7  48.9  233.8
Baking Goods           123.8  34.4  258.4
Breads                 144.1  33.0  265.5
Breakfast              141.0  39.3  240.2
CANNED                 139.5  38.0  260.9
Canned                 136.5  37.1  269.4
DAIRY                  148.1  33.1  271.8
Dairy                  144.9  37.0  270.0
FROZEN FOODS           142.1  32.6  265.8
FRUITS AND VEGETABLES  140.7  36.9  266.8
Frozen Foods           131.0  31.5  270.6
Fruits and Vegetables  141.1  36.6  267.1
HARD DRINKS            144.9  36.7  264.2
HEALTH AND HYGIENE     117.6  33.1  258.8
HOUSEHOLD              149.8  33.0  268.0
Hard Drinks            142.8  33.8  260.4
Health and Hygiene     133.7  32.4  273.0
Household              149.9  32.8  268.7
MEAT                   140.7  39.0  257.0
Meat                   143.4  34.2

Prices cover the same range in every category and fat content does not change the average price. Price varies more within a category than between categories.

In [10]:
print('there are', training_data['store_code'].isna().sum(),
      'missing values in the store code column')
print(training_data['store_code'].value_counts())
print(training_data.groupby('store_code')['store_size'].nunique())

there are 0 missing values in the store code column
store_code
STORE-YLW    754
STORE-9RG    752
STORE-AGY    748
STORE-7WS    748
STORE-OYG    744
STORE-HL7    742
STORE-DKU    736
STORE-89Z    728
STORE-JOR    439
STORE-T5G    427
Name: count, dtype: int64
store_code
STORE-7WS    1
STORE-89Z    1
STORE-9RG    1
STORE-AGY    1
STORE-DKU    0
STORE-HL7    1
STORE-JOR    0
STORE-OYG    0
STORE-T5G    1
STORE-YLW    1
Name: store_size, dtype: int64


It was discovered that
1. there are no NaN store codes and there are only 10 different store codes.
2. 7 of the 10 stores have a consistent, single size; the remaining 3 (STORE-DKU, STORE-JOR, STORE-OYG) have no size value at all, so it cannot be recovered.

Hypotheses: "does a particular store sell a particular product", "does a particular store sell more of a product", "does a particular store have more sales".

In [11]:
print(training_data.groupby('store_code').agg(store_format=('store_format', 'first'),
                                              store_location_tier=('store_location_tier', 'first'),
                                              store_age_years=('store_age_years', 'nunique'),
                                              mean_sales=('total_sales', 'mean')).sort_values('mean_sales', ascending=False).round(0))

                    store_format store_location_tier  store_age_years  \
store_code                                                              
STORE-7WS   Flagship Hypermarket              Tier_3                1   
STORE-9RG   Standard Supermarket              Tier_2                1   
STORE-89Z   Standard Supermarket              Tier_1                1   
STORE-OYG   Standard Supermarket              Tier_2                1   
STORE-AGY   Standard Supermarket              Tier_3                1   
STORE-YLW   Standard Supermarket              Tier_1                1   
STORE-DKU   Standard Supermarket              Tier_2                1   
STORE-HL7             Superstore              Tier_3                1   
STORE-T5G            Corner Shop              Tier_1                1   
STORE-JOR            Corner Shop              Tier_3                1   

            mean_sales  
store_code              
STORE-7WS       3660.0  
STORE-9RG       2479.0  
STORE-89Z       2366.0 

Each store code has exactly one format and one tier, and one store age. So store_format, store_location_tier, store_size and store_age_years are all just different names for the same 10 stores. I keep this in mind for the features later.

In [12]:
sns.boxplot(data=training_data, x='store_code', y='total_sales')
plt.xticks(rotation=45)
plt.title('Total Sales Distribution by Store')
plt.show()
print(training_data.groupby('store_code')['total_sales'].agg(['mean', 'min', 'max']))

                   mean     min       max
store_code                               
STORE-7WS   3660.182219  243.24  12996.82
STORE-89Z   2365.926745  155.47   7477.67
STORE-9RG   2479.168098  117.76   8726.50
STORE-AGY   2254.890120   72.28   8141.49
STORE-DKU   2177.136481  147.20   9388.15
STORE-HL7   1971.661954   73.54   7016.22
STORE-JOR    334.435011   33.30   1830.37
STORE-OYG   2365.833656  149.73  10130.90
STORE-T5G    338.326651   32.70   1523.62
STORE-YLW   2253.950690  164.83   9999.51


C:\Users\OPEYEMI\AppData\Local\Temp\ipykernel_44332\2988385770.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Sales vary noticeably by store:**
some stores consistently sell more regardless of product, so store_code itself is a meaningful signal, not just a label. The two corner shops (STORE-JOR and STORE-T5G) sell about 6 to 7 times less than the other stores, and STORE-7WS (the flagship hypermarket) sells the most.

In [13]:
#seeing how store size affects total sales

training_data.groupby('store_size', dropna=False)['total_sales'].agg(['mean', 'min', 'max'])

,mean,min,max
store_size,,,
Large,2254.890120,72.28,8141.49
Medium,2670.506826,73.54,12996.82
Small,1918.405954,32.70,9999.51
NaN,1828.749171,33.30,10130.90


Contrary to what you would expect, store size doesn't scale predictably with sales: medium stores actually outsell large stores on average. This breaks the intuitive assumption that "bigger store = more sales". Size is just attached to particular stores, so store_format and store_code capture the real difference and store_size on its own is not a strong predictor.

In [14]:
#to check what product categories sell more..
training_data.groupby('product_category')['total_sales'].agg(['mean', 'min', 'max'])

,mean,min,max
product_category,,,
BAKING GOODS,2051.639781,93.33,7720.12
BREADS,1853.545778,88.73,5897.21
BREAKFAST,1996.679474,109.66,7125.94
Baking Goods,1916.820039,36.15,6745.85
Breads,2292.078000,34.56,9258.66
Breakfast,1827.852195,39.63,6405.06
CANNED,2231.389915,83.29,8247.44
Canned,2127.398229,41.63,7556.65
DAIRY,2124.938671,40.12,7778.41


Looking at the mean there are no notable conclusions to draw.
product category has to be cleaned due to inconsistencies in the naming..

In [15]:
#inspecting the product price and total sales

plt.figure(figsize=(8, 5))
plt.scatter(training_data['product_price'], training_data['total_sales'], alpha=0.3, s=10)

z = np.polyfit(training_data['product_price'], training_data['total_sales'], 1)
x_line = np.linspace(training_data['product_price'].min(), training_data['product_price'].max(), 100)
plt.plot(x_line, z[0]*x_line + z[1], color='red', linewidth=2, label='Trend line')

plt.xlabel('Product Price')
plt.ylabel('Total Sales')
plt.title('Product Price vs. Total Sales')
plt.legend()
plt.show()
print('the correlation between the total sales and product price is',
      training_data['product_price'].corr(training_data['total_sales']))

the correlation between the total sales and product price is 0.5698334561303245


C:\Users\OPEYEMI\AppData\Local\Temp\ipykernel_44332\3210064078.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Product price shows a real, positive relationship with total sales. The correlation is approximately 0.57: as price increases, total sales tends to increase as well. This is a meaningfully stronger relationship than any other single numeric feature. However, it is far from perfect. The spread around the trend line gets wider as the price goes up, so price alone does not determine total_sales; the store is the other big factor.

In [16]:
#studying store location tier, store format as this could be factors that affect total sales..

print(training_data.groupby('store_location_tier')['total_sales'].agg(['mean', 'min', 'max']))
print(training_data.groupby('store_format')['total_sales'].agg(['mean', 'min', 'max']))
print(training_data['store_size'].isna().value_counts())

                            mean     min       max
store_location_tier                               
Tier_1               1868.171278   32.70   9999.51
Tier_2               2341.795296  117.76  10130.90
Tier_3               2254.114400   33.30  12996.82
                             mean     min       max
store_format                                       
Corner Shop            336.353868   32.70   1830.37
Flagship Hypermarket  3660.182219  243.24  12996.82
Standard Supermarket  2316.319677   72.28  10130.90
Superstore            1971.661954   73.54   7016.22
store_size
False    4899
True     1919
Name: count, dtype: int64


On the effect on total sales
1. tier 2 stores sell more than tier 3 and tier 1
2. flagship hypermarkets sell more than standard supermarkets, superstores and corner shops (superstores sell a little less than standard supermarkets)

Also, 3 different store codes have no store size all through the data. These are codes DKU, OYG and JOR.

## 4. Encoding for models that need it

Random Forest cannot read text, so it needs encoded data. The store location tier and the store format are ordinal encoded because of the ordering I noticed in the sales, and the other categories are one hot encoded.

Note: the corner shop label in the data is `corner shop` (with a space). My first mapping used `cornershop`, which silently turned every corner shop into a missing value, so I fixed it and added a check.

In [17]:
encode_columns = ['fat_content', 'product_category', 'store_code', 'store_size']

train_encoded = training_data.copy()
test_encoded = test_data.copy()

#Ordinal Encoding because of the noticed ordering of the categories
tier_encoding = {
    'tier_1': 1,
    'tier_2': 3,
    'tier_3': 2
}

store_format_encoding = {
    'flagship hypermarket': 4,
    'standard supermarket': 3,
    'superstore': 2,
    'corner shop': 1
}

In [18]:
#data cleaning for models that need encoding

for cc in [train_encoded, test_encoded]:
    cc['store_location_tier'] = cc['store_location_tier'].str.strip().str.lower()
    cc['store_location_tier'] = cc['store_location_tier'].map(tier_encoding)
    cc['store_format'] = cc['store_format'].str.strip().str.lower().map(store_format_encoding)
    cc['store_size'] = cc['store_size'].transform(
        lambda x: x.fillna('unknown').str.strip().str.lower()
    )
    cc['product_category'] = cc['product_category'].str.lower().str.strip()

for cc in [train_encoded, test_encoded]:
    assert cc['store_location_tier'].notna().all() and cc['store_format'].notna().all()

In [19]:
#encoding necessary categorical columns for models that need encoding

train_encoded = pd.get_dummies(train_encoded, columns=encode_columns, dtype=int)
test_encoded = pd.get_dummies(test_encoded, columns=encode_columns, dtype=int)
test_encoded = test_encoded.reindex(columns=train_encoded.drop(columns='total_sales').columns, fill_value=0)

print(train_encoded.shape, test_encoded.shape)

(6818, 41) (1705, 40)


## 5. Data cleaning and feature engineering

This is the cleaning for the models that do not need encoding (CatBoost, LightGBM, XGBoost).

In my first version I did the cleaning and the feature engineering separately on train and on test. That was a mistake for the features that count things: `product_num_stores` averaged 4.4 in train but only 1.6 in test, so the model learned something that changed meaning at prediction time. Now train and test are stacked together, cleaned and given features in one go (no feature here uses `total_sales`), then split back.

In [20]:
all_data = pd.concat([training_data.assign(part='train'), test_data.assign(part='test')], ignore_index=True)
cc = all_data

cc['product_category'] = cc['product_category'].str.lower().str.strip()

cc['store_format'] = cc['store_format'].str.lower().str.strip()

cc['store_location_tier'] = cc['store_location_tier'].str.lower().str.strip()

cc['fat_content'] = cc['fat_content'].str.lower().str.strip()

cc['store_size'] = cc.groupby('store_code')['store_size'].transform(
    lambda x: x.fillna(x.mode().iloc[0]) if x.notna().any() else x.fillna('unknown')
)

cc['store_size'] = cc['store_size'].str.strip().str.lower().fillna('unknown')

cc['product_weight_kg'] = cc.groupby('product_code')['product_weight_kg'].transform(
    lambda x: x.fillna(x.mean())
)

cc['product_weight_kg'] = cc['product_weight_kg'].fillna(cc['product_weight_kg'].mean())

cc['shelf_visibility'] = cc['shelf_visibility'].replace(0, np.nan)

cc['shelf_visibility'] = cc.groupby('product_code')['shelf_visibility'].transform(
    lambda x: x.fillna(x.mean())
)

cc['shelf_visibility'] = cc.groupby('product_category')['shelf_visibility'].transform(
    lambda x: x.fillna(x.mean())
)

cc['shelf_visibility'] = cc['shelf_visibility'].fillna(cc['shelf_visibility'].mean())

print(cc.drop(columns='total_sales').isna().sum())

id                     0
product_code           0
product_weight_kg      0
fat_content            0
shelf_visibility       0
product_category       0
product_price          0
store_code             0
store_age_years        0
store_size             0
store_location_tier    0
store_format           0
part                   0
dtype: int64


Store size is filled with the most common size of the store. The 3 stores with no size at all get "unknown". A shelf visibility of 0 is not realistic, so it is treated as missing and filled from the same product, then from the same category.

**Feature engineering**

These are the features I tried in my first version. I tested each one on its own with CatBoost, and kept the ones that did not make the score worse.

In [21]:
#feature engineering..

#-the count of how many different stores carry a given product.
cc['product_num_stores'] = cc.groupby('product_code')['store_code'].transform('nunique')

#-product average price
cc['product_avg_price'] = cc.groupby('product_code')['product_price'].transform('mean')

#-visibility ratio to category average
cc['visibility_ratio_to_cat'] = cc['shelf_visibility'] / (
    cc.groupby('product_category')['shelf_visibility'].transform('mean') + 1e-6
)

#-store format and location tier combination
cc['store_format_tier'] = cc['store_format'] + "_" + cc['store_location_tier']

training_data = all_data[all_data['part'] == 'train'].drop(columns='part').reset_index(drop=True)
test_data = all_data[all_data['part'] == 'test'].drop(columns=['part', 'total_sales']).reset_index(drop=True)

print('product_num_stores  train:', round(training_data['product_num_stores'].mean(), 2),
      '| test:', round(test_data['product_num_stores'].mean(), 2))
print(training_data.isna().sum().sum(), 'missing values left in train')
training_data.head()

product_num_stores  train: 5.91 | test: 5.83
0 missing values left in train


,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,total_sales,product_num_stores,product_avg_price,visibility_ratio_to_cat,store_format_tier
0,row_00000,PRD-PRFP9S,14.252000,low fat,0.0271,frozen foods,81.37,STORE-AGY,45,large,tier_3,standard supermarket,1764.98,6,78.20500,0.385613,standard supermarket_tier_3
1,row_00001,PRD-PXXK71,7.698000,low fat,0.0720,health and hygiene,42.05,STORE-YLW,35,small,tier_1,standard supermarket,342.13,5,44.57200,1.204107,standard supermarket_tier_1
2,row_00002,PRD-V5MOIJ,14.264000,regular,0.0421,canned,41.35,STORE-89Z,33,medium,tier_1,standard supermarket,378.85,8,41.68375,0.586626,standard supermarket_tier_1
3,row_00003,PRD-UN5Z3J,12.899667,regular,0.0449,soft drinks,174.35,STORE-7WS,47,medium,tier_3,flagship hypermarket,5595.72,5,175.81200,0.652831,flagship hypermarket_tier_3
4,row_00004,PRD-6RDQYB,10.338000,regular,0.0120,meat,203.06,STORE-9RG,28,small,tier_2,standard supermarket,2375.36,7,195.96000,0.182796,standard supermarket_tier_2


Features I dropped from my first version, and why
1. `broad_category` (first two letters of the product code): it is `PR` on every row, so it carries no information.
2. `store_format_size` and `store_product_variety`: they only rename the store code, since each store has one format and one size.
3. `price_per_kg`, `price_diff_from_cat_mean` and `price_ratio_to_cat_mean`: on their own they made CatBoost slightly worse, not better.

## 6. CatBoost

Here I train a model on the categorical data: CatBoost.

The cell below trains a CatBoostRegressor using 10-fold cross-validation. The dataset is shuffled and split into 10 folds, with each fold used once as the validation set while the remaining folds are used for training. RMSE is calculated for each fold, and the mean and standard deviation of the fold scores are used to evaluate the model's overall performance and consistency.

One change from my first version: I no longer use the validation fold for early stopping. Doing that lets the model pick its stopping point using the same rows that are then scored, so the CV score looks better than it really is. All models below use the same folds, so their scores can be compared directly.

In [22]:
X = training_data.drop(['id', 'total_sales', 'product_code'], axis=1)

y = training_data['total_sales']

test_x = test_data.drop(['id', 'product_code'], axis=1)

categorical_columns = [
    'product_category',
    'fat_content',
    'store_format',
    'store_location_tier',
    'store_size',
    'store_code',
    'store_format_tier'
]

print(list(X.columns))
print(list(X.columns) == list(test_x.columns))

['product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'product_num_stores', 'product_avg_price', 'visibility_ratio_to_cat', 'store_format_tier']
True


In [23]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

kf = KFold(n_splits=10, shuffle=True, random_state=42)
folds = list(kf.split(X))
results = {}

cat_params = dict(
    iterations=500,
    learning_rate=0.03,
    depth=3,
    l2_leaf_reg=3,
    min_data_in_leaf=10,
    bagging_temperature=1,
    boosting_type='Ordered',
    cat_features=categorical_columns,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose=False,
    random_state=42
)

rmse_scores_cb = []

for fold, (train_idx, val_idx) in enumerate(folds):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_cb = CatBoostRegressor(**cat_params)
    model_cb.fit(X_train, y_train)

    preds = model_cb.predict(X_val)
    fold_rmse = mean_squared_error(y_val, preds) ** 0.5
    rmse_scores_cb.append(fold_rmse)
    print(f'Fold {fold+1} RMSE: {fold_rmse:.2f}')

rmse_scores_cb = np.array(rmse_scores_cb)
print(f'Mean RMSE: {rmse_scores_cb.mean():.2f}')
print(f'Std RMSE: {rmse_scores_cb.std():.2f}')
results['CatBoost'] = rmse_scores_cb.mean()

Fold 1 RMSE: 1111.89
Fold 2 RMSE: 1016.36
Fold 3 RMSE: 1046.31
Fold 4 RMSE: 1060.01
Fold 5 RMSE: 1100.42
Fold 6 RMSE: 1074.30
Fold 7 RMSE: 1117.32
Fold 8 RMSE: 1113.90
Fold 9 RMSE: 1087.40
Fold 10 RMSE: 1010.12
Mean RMSE: 1073.80
Std RMSE: 37.61


CatBoost gets a mean RMSE of about **1073.8**, with a standard deviation of about 37.6 across the 10 folds. That fold-to-fold spread is large compared to the differences between models later, so a 1 or 2 point change in RMSE between two models is not something to read too much into.

In [24]:
cat_model = CatBoostRegressor(**cat_params)
cat_model.fit(X, y)

test_predictions_cat = cat_model.predict(test_x)

pd.DataFrame(
    {
        'id': test_data['id'],
        'total_sales': test_predictions_cat
    }
).to_csv(
    'submission_cat.csv',
    index=False
)

In [25]:
importance = pd.Series(cat_model.get_feature_importance(), index=X.columns).sort_values()

plt.figure(figsize=(7, 4.5))
plt.barh(importance.index, importance.values)
plt.title('CatBoost feature importance')
plt.xlabel('importance (%)')
plt.tight_layout()
plt.show()

C:\Users\OPEYEMI\AppData\Local\Temp\ipykernel_44332\3257117297.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`product_price` and `store_code` carry almost all of the importance, and the rest of the store columns (`store_format`, `store_location_tier`, `store_format_tier`, `store_size`, `store_age_years`) share what is left, because they all describe the same 10 stores. This matches what I found in section 3: price and the identity of the store are what actually drive sales.

## 7. LightGBM

Now I am going to be using another model - LightGBM. It uses the same folds as CatBoost.

In [26]:
import lightgbm as lgb

X_lgb = X.copy()
test_x_lgb = test_x.copy()

for col in categorical_columns:
    cats = sorted(all_data[col].unique())
    X_lgb[col] = pd.Categorical(X_lgb[col], categories=cats)
    test_x_lgb[col] = pd.Categorical(test_x_lgb[col], categories=cats)

rmse_scores_lgb = []

for fold, (train_idx, val_idx) in enumerate(folds):
    X_train, X_val = X_lgb.iloc[train_idx], X_lgb.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_lgb = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.028,
        max_depth=4,
        num_leaves=12,
        reg_alpha=4,
        reg_lambda=3,
        random_state=42,
        verbosity=-1
    )
    model_lgb.fit(X_train, y_train)

    preds = model_lgb.predict(X_val)
    fold_rmse = mean_squared_error(y_val, preds) ** 0.5
    rmse_scores_lgb.append(fold_rmse)
    print(f'Fold {fold+1} RMSE: {fold_rmse:.2f}')

rmse_scores_lgb = np.array(rmse_scores_lgb)
print(f'Mean RMSE: {rmse_scores_lgb.mean():.2f}')
print(f'Std RMSE: {rmse_scores_lgb.std():.2f}')
results['LightGBM'] = rmse_scores_lgb.mean()

Fold 1 RMSE: 1125.28
Fold 2 RMSE: 1040.46
Fold 3 RMSE: 1038.04
Fold 4 RMSE: 1080.41
Fold 5 RMSE: 1103.73
Fold 6 RMSE: 1086.33
Fold 7 RMSE: 1142.14
Fold 8 RMSE: 1110.12
Fold 9 RMSE: 1099.13
Fold 10 RMSE: 1015.04
Mean RMSE: 1084.07
Std RMSE: 38.94


LightGBM gets a mean RMSE of about **1084.0**, a bit behind CatBoost. On this dataset CatBoost handles the categorical columns better.

## 8. Random Forest

I also want to try the Random Forest regressor. It uses the encoded data from section 4.

In [27]:
from sklearn.ensemble import RandomForestRegressor

X_rf = train_encoded.drop(['id', 'total_sales', 'product_code'], axis=1)
y_rf = train_encoded['total_sales']

rmse_scores_rf = []

for fold, (train_idx, val_idx) in enumerate(folds):
    train_val_rf, test_val_rf = X_rf.iloc[train_idx], X_rf.iloc[val_idx]
    train_y_rf, test_y_rf = y_rf.iloc[train_idx], y_rf.iloc[val_idx]

    model_rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=5,
        min_samples_split=3,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    model_rf.fit(train_val_rf, train_y_rf)

    preds = model_rf.predict(test_val_rf)
    fold_rmse = mean_squared_error(test_y_rf, preds) ** 0.5
    rmse_scores_rf.append(fold_rmse)
    print(f'Fold {fold+1} RMSE: {fold_rmse:.2f}')

rmse_scores_rf = np.array(rmse_scores_rf)
print(f'Mean RMSE: {rmse_scores_rf.mean():.2f}')
print(f'Std RMSE: {rmse_scores_rf.std():.2f}')
results['Random Forest'] = rmse_scores_rf.mean()

Fold 1 RMSE: 1123.89
Fold 2 RMSE: 1031.84
Fold 3 RMSE: 1059.88
Fold 4 RMSE: 1068.10
Fold 5 RMSE: 1105.64
Fold 6 RMSE: 1084.90
Fold 7 RMSE: 1130.31
Fold 8 RMSE: 1116.19
Fold 9 RMSE: 1103.52
Fold 10 RMSE: 1015.20
Mean RMSE: 1083.95
Std RMSE: 37.30


## 9. XGBoost

XGBoost was used in my stacking cell before but I never imported it, so here it gets its own proper cross validation first.

In [28]:
from xgboost import XGBRegressor

def make_xgb():
    return XGBRegressor(
        n_estimators=200,
        learning_rate=0.020,
        max_depth=3,
        reg_alpha=3,
        reg_lambda=3,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        enable_categorical=True,
        tree_method='hist',
        random_state=42
    )

rmse_scores_xgb = []

for fold, (train_idx, val_idx) in enumerate(folds):
    X_train, X_val = X_lgb.iloc[train_idx], X_lgb.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_xgb = make_xgb()
    model_xgb.fit(X_train, y_train)

    preds = model_xgb.predict(X_val)
    fold_rmse = mean_squared_error(y_val, preds) ** 0.5
    rmse_scores_xgb.append(fold_rmse)
    print(f'Fold {fold+1} RMSE: {fold_rmse:.2f}')

rmse_scores_xgb = np.array(rmse_scores_xgb)
print(f'Mean RMSE: {rmse_scores_xgb.mean():.2f}')
print(f'Std RMSE: {rmse_scores_xgb.std():.2f}')
results['XGBoost'] = rmse_scores_xgb.mean()

Fold 1 RMSE: 1110.27
Fold 2 RMSE: 1038.19
Fold 3 RMSE: 1035.07
Fold 4 RMSE: 1071.74
Fold 5 RMSE: 1094.23
Fold 6 RMSE: 1077.97
Fold 7 RMSE: 1122.82
Fold 8 RMSE: 1112.16
Fold 9 RMSE: 1098.01
Fold 10 RMSE: 1000.83
Mean RMSE: 1076.13
Std RMSE: 37.78


## 10. Stacking

Now I perform stacking. Each model makes out of fold predictions (every row is predicted by a model that never saw it), and a Ridge model learns how to combine them.

In my first version the Ridge model was scored on the same rows it was trained on, which looks better than it really is. Here its score is calculated with cross validation.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict

oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(folds):
    y_train = y.iloc[train_idx]

    m_cat = CatBoostRegressor(**cat_params)
    m_cat.fit(X.iloc[train_idx], y_train)
    oof_cat[val_idx] = m_cat.predict(X.iloc[val_idx])

    m_xgb = make_xgb()
    m_xgb.fit(X_lgb.iloc[train_idx], y_train)
    oof_xgb[val_idx] = m_xgb.predict(X_lgb.iloc[val_idx])

    m_lgb = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.028, max_depth=4, num_leaves=12,
                              reg_alpha=4, reg_lambda=3, random_state=42, verbosity=-1)
    m_lgb.fit(X_lgb.iloc[train_idx], y_train)
    oof_lgb[val_idx] = m_lgb.predict(X_lgb.iloc[val_idx])

def rmse(a, b):
    return mean_squared_error(a, b) ** 0.5

print(f"CatBoost alone RMSE:  {rmse(y, oof_cat):.2f}")
print(f"XGBoost alone RMSE:   {rmse(y, oof_xgb):.2f}")
print(f"LightGBM alone RMSE:  {rmse(y, oof_lgb):.2f}")

naive_avg = (oof_cat + oof_xgb + oof_lgb) / 3
print(f"\nNaive 3-way average RMSE: {rmse(y, naive_avg):.2f}")

meta_X = np.column_stack([oof_cat, oof_xgb, oof_lgb])
meta_model = Ridge(alpha=1.0)

stacked_preds = cross_val_predict(meta_model, meta_X, y, cv=KFold(5, shuffle=True, random_state=1))
print(f"Stacked (Ridge) RMSE, scored with cross validation: {rmse(y, stacked_preds):.2f}")

meta_model.fit(meta_X, y)
print(f"Learned weights -> CatBoost: {meta_model.coef_[0]:.3f}, XGBoost: {meta_model.coef_[1]:.3f}, LightGBM: {meta_model.coef_[2]:.3f}")
print(f"Intercept: {meta_model.intercept_:.2f}")
results['Stacking (Ridge)'] = rmse(y, stacked_preds)

On their own: CatBoost RMSE 1074.5, XGBoost 1077.2, LightGBM 1084.8. The plain average of the three gives 1075.9, and the Ridge stack, scored properly this time, gives 1074.1. That is only a small improvement over CatBoost by itself, and the learned weight on LightGBM is negative, so the stack is really leaning on CatBoost and XGBoost.

## 11. Summary of results

In [ ]:
summary = pd.Series(results, name='CV RMSE').sort_values().round(2)
print(summary)

CatBoost            1073.80
Stacking (Ridge)    1074.09
XGBoost             1076.51
Random Forest       1083.95
LightGBM            1084.07
Name: CV RMSE, dtype: float64


CatBoost and the Ridge stack are the best of the models I tried, both close to 1074. LightGBM and Random Forest trail behind. The gap between the best and worst here (about 10 points) is smaller than the standard deviation of a single model across folds (about 37 to 39 points), which tells me the models are all picking up roughly the same signal: price and store. The submission file is `submission_cat.csv`, from the CatBoost model trained on all the training rows.